# Likelihood derivation

Stratum $i$ has rows $j = 1, \dots, n_i$, and deaths in row $j$ are Poisson with mean

$$\lambda_{ij}(\alpha_i, \beta) = p_{ij} \exp(\alpha_i) \phi_{ij}(\beta)$$

where $p_{ij}$ is person-years, $\exp(\alpha_i)$ is the baseline rate of the stratum, and $\phi_{ij} = 1 + \text{ERR}$ is the relative risk. We integrate out $\alpha_i$ so there's no baseline to estimate for every stratum

In [1]:
from sympy import (
    Function,
    Idx,
    IndexedBase,
    Integral,
    Product,
    Sum,
    exp,
    expand,
    expand_func,
    expand_log,
    factor_terms,
    factorial,
    gamma,
    integrate,
    log,
    oo,
    prod,
    simplify,
    symbols,
)
from sympy.simplify.simplify import factor_sum

In [2]:
j = Idx("j")
n_i = symbols("n_i", integer=True, positive=True)
alpha_i, beta = symbols(r"\alpha_i \beta", real=True)
p = IndexedBase("p_i", positive=True)  # person-years
y = IndexedBase("y_i", integer=True, nonnegative=True)  # deaths
phi = Function(r"\phi_i", positive=True)(j, beta)

lmbda = p[j] * exp(alpha_i) * phi
ell = Product(lmbda ** y[j] / factorial(y[j]) * exp(-lmbda), (j, 1, n_i))
ell

Product((\phi_i(j, \beta)*exp(\alpha_i)*p_i[j])**y_i[j]*exp(-\phi_i(j, \beta)*exp(\alpha_i)*p_i[j])/factorial(y_i[j]), (j, 1, n_i))

Take the log so $\alpha_i$ can be pulled out of the sums, with $Y_i = \sum_j y_{ij}$ and $S_i = \sum_j \phi_{ij}(\beta) p_{ij}$

In [3]:
Y_i, S_i, A = symbols("Y_i S_i A", positive=True)
Y = Sum(y[j], (j, 1, n_i))
S = Sum(phi * p[j], (j, 1, n_i))

log_ell = factor_terms(expand(expand(log(ell)))).subs({Y: Y_i, S: S_i})
log_ell

-S_i*exp(\alpha_i) + Y_i*\alpha_i + Sum(log(\phi_i(j, \beta))*y_i[j], (j, 1, n_i)) + Sum(log(p_i[j])*y_i[j], (j, 1, n_i)) - Sum(log(factorial(y_i[j])), (j, 1, n_i))

Integrate over $\alpha_i$ with a flat prior, using $A = e^{\alpha_i}$, and take the log

In [4]:
marginal = (
    Integral(exp(log_ell), (alpha_i, -oo, oo))
    .transform(exp(alpha_i), A)
    .doit()
    .simplify()
)
log_marginal = expand(log(marginal))
log_marginal

-Y_i*log(S_i) + log(gamma(Y_i)) + Sum(log(\phi_i(j, \beta))*y_i[j] + log(p_i[j])*y_i[j], (j, 1, n_i)) - Sum(log(factorial(y_i[j])), (j, 1, n_i))

Replacing $\log\Gamma(Y_i)$ with $\log\Gamma(Y_i + 1)$ adds $\log Y_i$, which doesn't depend on $\beta$. The result is the log pmf of how the $Y_i$ deaths are split across the rows

In [5]:
modified = log_marginal - log(gamma(Y_i)) + log(gamma(Y_i + 1))
simplify(expand_func(modified - log_marginal))

log(Y_i)

With no deaths in the stratum the original is infinite, since $\Gamma(0)$ is, and the modified version is 0. So strata without deaths drop out

In [6]:
no_deaths = {Y_i: 0}  # and every y_ij = 0, so the sums vanish
(
    log_marginal.subs(no_deaths).replace(Sum, lambda *args: 0),
    modified.subs(no_deaths).replace(Sum, lambda *args: 0),
)

(zoo, 0)

To avoid overflow in $S_i$, write $w_{ij} = \log p_{ij} + \log\phi_{ij}(\beta)$ and subtract a constant $c$ (the mean of $w_{ij}$ over the stratum) before exponentiating. The shifted log likelihood is the same as the modified one (difference is 0)

In [7]:
c = symbols("c", real=True)
w = log(p[j]) + log(phi)

shifted = (
    Sum(y[j] * (w - c), (j, 1, n_i))
    - Sum(log(factorial(y[j])), (j, 1, n_i))
    - Y * log(Sum(exp(w - c), (j, 1, n_i)))
    + log(gamma(Y + 1))
)
difference = (shifted - modified.subs({Y_i: Y, S_i: S})).replace(
    lambda e: isinstance(e, Sum), lambda e: factor_sum(e.expand())
)
simplify(factor_terms(expand(expand_log(difference, force=True))))

0

## Check

Three rows with deaths $(2, 0, 3)$. Integrating directly should match the closed form

In [8]:
a = symbols("a", real=True)
ps = symbols("p1:4", positive=True)
fs = symbols("phi1:4", positive=True)
ys = [2, 0, 3]

ell_concrete = prod(
    (ps[k] * exp(a) * fs[k]) ** ys[k] / factorial(ys[k]) * exp(-ps[k] * exp(a) * fs[k])
    for k in range(3)
)
direct = integrate(ell_concrete.subs(exp(a), A) / A, (A, 0, oo), conds="none")

closed_form = (
    marginal.subs({Y_i: sum(ys), S_i: sum(ps[k] * fs[k] for k in range(3)), n_i: 3})
    .doit()
    .subs({p[k + 1]: ps[k] for k in range(3)})
    .subs({y[k + 1]: ys[k] for k in range(3)})
    .subs({phi.subs(j, k + 1): fs[k] for k in range(3)})
)
simplify(direct - closed_form)

0